# Day 1: Infrastructure Foundation & Iceberg Storage Setup

This notebook simulates the creation of a local Iceberg catalog and creates the Bronze raw payload schema with audit metadata.

In [ ]:
!pip install pyspark pyiceberg duckdb splink


In [ ]:
import os
from pyspark.sql import SparkSession

# Setting SPARK_VERSION environment variable (required by PyDeequ usage patterns eventually)
os.environ['SPARK_VERSION'] = '3.3'

# Create SparkSession configured for local Iceberg catalog
spark = SparkSession.builder \
    .appName("Day01_Iceberg_Setup") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.3_2.12:1.3.1,org.apache.iceberg:iceberg-aws-bundle:1.3.1") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "/tmp/ehdip_warehouse") \
    .getOrCreate()

print("SparkSession initialized with Iceberg capabilities.")

In [ ]:
# Create the Bronze database (namespace)
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.bronze_db")

# Create the Bronze raw payload table with audit metadata
spark.sql("""
CREATE TABLE IF NOT EXISTS local.bronze_db.raw_payloads (
    payload_id STRING COMMENT 'Unique identifier for the incoming payload',
    source_system_id STRING COMMENT 'Identifier of the source system (e.g., EHR_Epic, EDI_Clearinghouse)',
    ingestion_timestamp TIMESTAMP COMMENT 'Time the record was ingested into the platform',
    raw_payload_json STRING COMMENT 'The original raw JSON payload'
)
USING iceberg
PARTITIONED BY (days(ingestion_timestamp))
""")

print("Bronze raw payload Iceberg schema created successfully.")

In [ ]:
# Verify schema
spark.sql("DESCRIBE TABLE local.bronze_db.raw_payloads").show(truncate=False)